# Smart Waste Collection Robot Agent — Bengaluru Smart City Initiative
## Using IDA* (Iterative Deepening A*) Informed Search Algorithm

**Objective:** Design and implement a Smart Waste Collection Robot Agent that uses the IDA* algorithm
to find the least-cost route from a starting waste collection center to a target waste processing unit
in Bengaluru's waste management network.

**Problem Formulation:**
- **State Space:** Set of all waste collection centers and processing units in the city
- **Initial State:** Starting location (user-specified)
- **Goal State:** Destination waste processing unit (user-specified)
- **Actions:** Move from one location to an adjacent location via a connecting road
- **Path Cost:** Sum of edge weights (travel cost / distance / fuel consumption) along the route
- **Solution:** A sequence of locations forming the least-cost path from source to destination

## 1. Import Required Libraries

In [37]:
import heapq
import time
from collections import defaultdict, deque

# Constant representing infinity (used as initial threshold in IDA*)
INF = float('inf')

## 2. Define the City Waste Management Graph

The city's waste management network is modeled as a **weighted undirected graph**:
- **Vertices (Nodes):** Waste collection centers or processing units
- **Edges:** Roads connecting those locations
- **Edge Weights:** Travel cost (distance or fuel consumption)

The `WasteManagementGraph` class provides:
- `add_edge()` — Add a bidirectional road between two locations with a given cost
- `get_neighbors()` — Retrieve all adjacent locations and their costs
- Node validation and basic error handling
- Battery/fuel capacity constraint for the robot agent

In [38]:
class WasteManagementGraph:
    """
    Weighted undirected graph representing Bengaluru's waste management network.
    Vertices = waste collection centers / processing units
    Edges    = roads with travel cost as weight
    """

    def __init__(self, battery_capacity=INF):
        self.graph = defaultdict(list)   # adjacency list: node -> [(neighbor, cost)]
        self.nodes = set()               # all registered nodes
        self.battery_capacity = battery_capacity  # max travel budget for the robot

    def add_edge(self, u, v, weight):
        """Add a bidirectional edge between u and v with the given weight."""
        if weight < 0:
            raise ValueError(f"Edge weight must be non-negative, got {weight}")
        self.graph[u].append((v, weight))
        self.graph[v].append((u, weight))
        self.nodes.add(u)
        self.nodes.add(v)

    def get_neighbors(self, node):
        """Return the list of (neighbor, cost) pairs for a given node."""
        if node not in self.nodes:
            raise KeyError(f"Node '{node}' is not a valid location in the network")
        return self.graph[node]

    def validate_node(self, node):
        """Check whether a node exists in the graph."""
        if node not in self.nodes:
            raise KeyError(f"Location '{node}' not found in the waste management network. "
                           f"Available locations: {sorted(self.nodes)}")
        return True

    def display(self):
        """Pretty-print the adjacency list."""
        print("=" * 60)
        print("  Waste Management Network — Adjacency List")
        print("=" * 60)
        for node in sorted(self.nodes):
            neighbors = ", ".join(f"{v} (cost={w})" for v, w in self.graph[node])
            print(f"  {node:20s} → {neighbors}")
        print("=" * 60)
        print(f"  Total locations : {len(self.nodes)}")
        print(f"  Total roads     : {sum(len(v) for v in self.graph.values()) // 2}")
        print(f"  Battery capacity: {self.battery_capacity}")
        print("=" * 60)

## 3. Define the Heuristic Function

### Heuristic Design — `h(n)`

For the IDA* algorithm to be **optimal**, the heuristic must be:
1. **Admissible** — It must never overestimate the true cost to reach the goal: $h(n) \leq h^*(n)$
2. **Consistent (Monotonic)** — For every node $n$ and successor $n'$: $h(n) \leq c(n, n') + h(n')$

**Strategy used:** Minimum hop count × Minimum edge cost

| Component | Description |
|-----------|-------------|
| **Minimum hop count** | The fewest number of edges (hops) needed to travel from node $n$ to the goal, computed via BFS |
| **Minimum edge cost** | The smallest edge weight across the entire graph |
| **Heuristic formula** | $h(n) = \text{min\_hops}(n, \text{goal}) \times \min_{e \in E}(w(e))$ |

**Justification — Admissibility proof:**
- Any path from $n$ to the goal must traverse **at least** `min_hops` edges (BFS gives the shortest path in terms of edge count).
- Each edge costs **at least** `min_edge_cost`.
- Therefore, actual path cost $\geq$ `min_hops` $\times$ `min_edge_cost` $= h(n)$.
- Since $h(n) \leq h^*(n)$ always holds, the heuristic is **admissible**.

**Consistency:** For any edge $(n, n')$ with cost $c$:
$h(n) \leq c(n,n') + h(n')$ because removing one hop reduces the hop count by at most 1, and $c \geq \text{min\_edge\_cost}$.

In [39]:
def heuristic(graph, node, goal):
    """
    Compute the heuristic estimate h(n) from 'node' to 'goal'.

    Formula: h(n) = min_hop_count(n, goal) × min_edge_cost(graph)

    Admissibility proof:
        Any path from n to goal uses at least min_hop_count edges,
        and each edge costs at least min_edge_cost.
        Therefore, actual cost >= min_hop_count × min_edge_cost = h(n),
        so h(n) <= h*(n) always holds.
    """
    if node == goal:
        return 0

    # Find the minimum edge cost in the entire graph
    min_cost = INF
    for n in graph.nodes:
        for _, cost in graph.graph[n]:
            if cost < min_cost:
                min_cost = cost

    # BFS to find minimum hop count from node to goal
    visited = {node}
    queue = deque([(node, 0)])
    while queue:
        current, hops = queue.popleft()
        if current == goal:
            return hops * min_cost
        for neighbor, _ in graph.graph[current]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, hops + 1))

    return INF  # goal is unreachable

## 4. Implement the IDA* Algorithm

### Algorithm Overview

IDA* combines the optimality of A* with the linear memory usage of iterative deepening DFS:

1. Set the initial cost threshold $t = f(start) = g(start) + h(start) = 0 + h(start)$
2. Perform a depth-first search, pruning any node where $f(n) = g(n) + h(n) > t$
3. If the goal is not found, set $t$ to the minimum $f$-value that exceeded the previous threshold
4. Repeat until the goal is found or no path exists

### Cost Function — `f(n) = g(n) + h(n)`

| Component | Meaning |
|-----------|---------|
| $g(n)$ | **Actual cost** from the start node to the current node $n$ (sum of edge weights along the path) |
| $h(n)$ | **Heuristic estimate** of the cost from $n$ to the goal (min hops × min edge cost) |
| $f(n)$ | **Total estimated cost** of the cheapest solution through $n$ |

**Optimality guarantee:** Since $h(n)$ is admissible, IDA* is guaranteed to find the optimal (least-cost) path.

In [40]:
class IDAStarSolver:
    """
    IDA* (Iterative Deepening A*) solver for finding the optimal
    least-cost path in a weighted graph.
    """

    def __init__(self, graph):
        if not isinstance(graph, WasteManagementGraph):
            raise TypeError("Expected a WasteManagementGraph instance")
        self.graph = graph
        self.nodes_explored = 0       # total nodes expanded across all iterations
        self.visited_sequence = []    # order in which nodes are explored
        self.solution_path = []       # optimal path once found
        self.solution_cost = INF      # cost of the optimal path

    def solve(self, start, goal):
        """
        Run IDA* from 'start' to 'goal'.
        Returns (path, cost) or (None, INF) if no path exists.
        """
        # --- Input validation ---
        self.graph.validate_node(start)
        self.graph.validate_node(goal)

        # --- Reset counters ---
        self.nodes_explored = 0
        self.visited_sequence = []
        self.solution_path = []
        self.solution_cost = INF

        # --- Initial threshold = h(start) ---
        threshold = heuristic(self.graph, start, goal)
        path = [start]
        iteration = 0

        while True:
            iteration += 1
            result = self._search(path, 0, threshold, goal)

            if result == "FOUND":
                return self.solution_path, self.solution_cost

            if result == INF:
                # No path exists to the goal
                return None, INF

            # Update threshold to the minimum f-value that exceeded the old one
            threshold = result

    def _search(self, path, g, threshold, goal):
        """
        Recursive depth-limited search used by IDA*.

        Parameters
        ----------
        path      : current path from start
        g         : cost from start to current node
        threshold : current cost limit
        goal      : target node

        Returns
        -------
        "FOUND" if goal reached, else the minimum f-value exceeding threshold
        """
        node = path[-1]
        f = g + heuristic(self.graph, node, goal)

        # Prune: f exceeds the current threshold
        if f > threshold:
            return f

        # Track exploration
        self.nodes_explored += 1
        self.visited_sequence.append(node)

        # Goal check
        if node == goal:
            self.solution_path = list(path)
            self.solution_cost = g
            return "FOUND"

        min_threshold = INF

        # Expand neighbors sorted by cost (for deterministic behavior)
        neighbors = sorted(self.graph.get_neighbors(node), key=lambda x: x[1])

        for neighbor, cost in neighbors:
            # Cycle detection: skip nodes already on the current path
            if neighbor in path:
                continue

            # Battery / fuel constraint check
            if g + cost > self.graph.battery_capacity:
                continue  # cannot afford this edge

            path.append(neighbor)
            result = self._search(path, g + cost, threshold, goal)

            if result == "FOUND":
                return "FOUND"

            if result < min_threshold:
                min_threshold = result

            path.pop()

        return min_threshold

    def print_results(self, start, goal):
        """Run IDA* and print a formatted report."""
        print("\n" + "=" * 60)
        print("  IDA* SEARCH — SMART WASTE COLLECTION ROBOT AGENT")
        print("=" * 60)
        print(f"  Source      : {start}")
        print(f"  Destination : {goal}")
        print("-" * 60)

        start_time = time.time()
        path, cost = self.solve(start, goal)
        elapsed = time.time() - start_time

        if path is None:
            print(f"\n  No path exists from '{start}' to '{goal}'.")
            print(f"  Nodes explored : {self.nodes_explored}")
        else:
            print(f"\n  Optimal Path Found!")
            print(f"  Path            : {' → '.join(path)}")
            print(f"  Total Cost      : {cost}")
            print(f"  Nodes Explored  : {self.nodes_explored}")
            print(f"  Search Time     : {elapsed:.6f} seconds")
            print(f"\n  Visited Sequence (exploration order):")
            print(f"    {' → '.join(self.visited_sequence)}")

        print("=" * 60)
        return path, cost

## 5. PEAS Analysis — Smart Waste Collection Robot Agent

| Component | Description |
|-----------|-------------|
| **Performance Measure** | Minimize total travel cost (fuel + distance); maximize waste collected per trip; minimize route time; stay within battery constraints |
| **Environment** | Bengaluru city road network modeled as a weighted undirected graph; partially observable (robot knows its current location and adjacent roads); deterministic (edge costs are fixed); static (graph does not change during a single trip); discrete (finite set of locations and roads) |
| **Actuators** | Wheels/motors for movement along roads; waste collection mechanism; GPS-based navigation system |
| **Sensors** | GPS for current location; odometer for distance traveled; fuel/battery gauge; road connectivity sensors; waste level sensors at collection points |

### Agent Type
- **Type:** Goal-based, utility-based agent
- **Search Strategy:** IDA* (informed search with iterative deepening)
- **Rationality:** The agent is rational — it selects the action that maximizes expected utility (minimum cost path) given its knowledge of the environment

## 6. Case 1 — MG_Road to Yelahanka (7 locations, 8 roads)

### Graph Structure
```
Locations: MG_Road, Electronic_City, Koramangala, Whitefield, Yelahanka, Jayanagar, Hebbal
Roads (8 edges with costs):
  MG_Road         ↔ Electronic_City : 2
  MG_Road         ↔ Koramangala     : 4
  Electronic_City ↔ Whitefield      : 2
  Whitefield      ↔ Yelahanka       : 4
  Whitefield      ↔ Jayanagar       : 6
  Jayanagar       ↔ Yelahanka       : 2
  Koramangala     ↔ Hebbal          : 3
  Hebbal          ↔ Yelahanka       : 5
```

In [41]:
# ============================================================
# CASE 1: Bengaluru locations — 7 nodes, 8 edges
# ============================================================

# Build the graph
graph1 = WasteManagementGraph(battery_capacity=50)

# Add 8 roads (bidirectional edges with travel cost) — as per assignment
graph1.add_edge("MG_Road",         "Electronic_City", 2)
graph1.add_edge("MG_Road",         "Koramangala",     4)
graph1.add_edge("Electronic_City", "Whitefield",      2)
graph1.add_edge("Whitefield",      "Yelahanka",       4)
graph1.add_edge("Whitefield",      "Jayanagar",       6)
graph1.add_edge("Jayanagar",       "Yelahanka",       2)
graph1.add_edge("Koramangala",     "Hebbal",          3)
graph1.add_edge("Hebbal",          "Yelahanka",       5)

# Display the graph
graph1.display()

# Define source and destination
source1 = "MG_Road"
destination1 = "Yelahanka"

# Run IDA*
solver1 = IDAStarSolver(graph1)
path1, cost1 = solver1.print_results(source1, destination1)

  Waste Management Network — Adjacency List
  Electronic_City      → MG_Road (cost=2), Whitefield (cost=2)
  Hebbal               → Koramangala (cost=3), Yelahanka (cost=5)
  Jayanagar            → Whitefield (cost=6), Yelahanka (cost=2)
  Koramangala          → MG_Road (cost=4), Hebbal (cost=3)
  MG_Road              → Electronic_City (cost=2), Koramangala (cost=4)
  Whitefield           → Electronic_City (cost=2), Yelahanka (cost=4), Jayanagar (cost=6)
  Yelahanka            → Whitefield (cost=4), Jayanagar (cost=2), Hebbal (cost=5)
  Total locations : 7
  Total roads     : 8
  Battery capacity: 50

  IDA* SEARCH — SMART WASTE COLLECTION ROBOT AGENT
  Source      : MG_Road
  Destination : Yelahanka
------------------------------------------------------------

  Optimal Path Found!
  Path            : MG_Road → Electronic_City → Whitefield → Yelahanka
  Total Cost      : 8
  Nodes Explored  : 7
  Search Time     : 0.000039 seconds

  Visited Sequence (exploration order):
    MG_Road →

## 7. Case 2 — A to E (5 locations, 6 roads)

### Graph Structure
```
Locations: A, B, C, D, E
Roads (6 edges with costs):
  A ↔ B : 3
  A ↔ C : 2
  B ↔ D : 4
  C ↔ D : 1
  C ↔ E : 7
  D ↔ E : 2
```

In [42]:
# ============================================================
# CASE 2: Abstract locations — 5 nodes, 6 edges
# ============================================================

# Build the graph
graph2 = WasteManagementGraph(battery_capacity=30)

# Add 6 roads — as per assignment
graph2.add_edge("A", "B", 3)
graph2.add_edge("A", "C", 2)
graph2.add_edge("B", "D", 4)
graph2.add_edge("C", "D", 1)
graph2.add_edge("C", "E", 7)
graph2.add_edge("D", "E", 2)

# Display the graph
graph2.display()

# Define source and destination
source2 = "A"
destination2 = "E"

# Run IDA*
solver2 = IDAStarSolver(graph2)
path2, cost2 = solver2.print_results(source2, destination2)

  Waste Management Network — Adjacency List
  A                    → B (cost=3), C (cost=2)
  B                    → A (cost=3), D (cost=4)
  C                    → A (cost=2), D (cost=1), E (cost=7)
  D                    → B (cost=4), C (cost=1), E (cost=2)
  E                    → C (cost=7), D (cost=2)
  Total locations : 5
  Total roads     : 6
  Battery capacity: 30

  IDA* SEARCH — SMART WASTE COLLECTION ROBOT AGENT
  Source      : A
  Destination : E
------------------------------------------------------------

  Optimal Path Found!
  Path            : A → C → D → E
  Total Cost      : 5
  Nodes Explored  : 10
  Search Time     : 0.000053 seconds

  Visited Sequence (exploration order):
    A → A → C → A → C → D → A → C → D → E


## 8. Alternate Modeling Approach — A* Algorithm & Performance Comparison

### Why A* as the alternate approach?

| Property | IDA* | A* |
|----------|------|-----|
| **Memory** | $O(bd)$ — linear in depth (stores only current path) | $O(b^d)$ — exponential (stores entire Open/Closed list) |
| **Time** | May re-expand nodes across iterations | Expands each node at most once (with consistent heuristic) |
| **Optimality** | Optimal with admissible heuristic | Optimal with admissible heuristic |
| **Best for** | Large state spaces with limited memory | Smaller graphs where memory is not a constraint |

where $b$ = branching factor, $d$ = depth of optimal solution.

**Key trade-off:** IDA* uses much less memory but may explore more nodes due to re-expansion across iterations. A* is faster in practice for small graphs but requires storing all explored states.

In [43]:
class AStarSolver:
    """
    Standard A* search algorithm for comparison with IDA*.
    Uses a priority queue (min-heap) to always expand the lowest f(n) node.
    """

    def __init__(self, graph):
        if not isinstance(graph, WasteManagementGraph):
            raise TypeError("Expected a WasteManagementGraph instance")
        self.graph = graph
        self.nodes_explored = 0
        self.visited_sequence = []
        self.solution_path = []
        self.solution_cost = INF

    def solve(self, start, goal):
        """
        Run A* from 'start' to 'goal'.
        Returns (path, cost) or (None, INF) if no path exists.
        """
        self.graph.validate_node(start)
        self.graph.validate_node(goal)

        self.nodes_explored = 0
        self.visited_sequence = []

        # Priority queue: (f_cost, g_cost, node, path)
        open_list = [(heuristic(self.graph, start, goal), 0, start, [start])]
        # Best known g-cost for each node
        best_g = {start: 0}

        while open_list:
            f, g, current, path = heapq.heappop(open_list)

            self.nodes_explored += 1
            self.visited_sequence.append(current)

            # Goal reached
            if current == goal:
                self.solution_path = path
                self.solution_cost = g
                return path, g

            for neighbor, cost in self.graph.get_neighbors(current):
                new_g = g + cost

                # Battery constraint
                if new_g > self.graph.battery_capacity:
                    continue

                # Only expand if we found a cheaper path to neighbor
                if neighbor not in best_g or new_g < best_g[neighbor]:
                    best_g[neighbor] = new_g
                    f_new = new_g + heuristic(self.graph, neighbor, goal)
                    heapq.heappush(open_list, (f_new, new_g, neighbor, path + [neighbor]))

        return None, INF

    def print_results(self, start, goal):
        """Run A* and print a formatted report."""
        print("\n" + "=" * 60)
        print("  A* SEARCH — ALTERNATE APPROACH")
        print("=" * 60)
        print(f"  Source      : {start}")
        print(f"  Destination : {goal}")
        print("-" * 60)

        start_time = time.time()
        path, cost = self.solve(start, goal)
        elapsed = time.time() - start_time

        if path is None:
            print(f"\n  No path exists from '{start}' to '{goal}'.")
            print(f"  Nodes explored : {self.nodes_explored}")
        else:
            print(f"\n  Optimal Path Found!")
            print(f"  Path            : {' → '.join(path)}")
            print(f"  Total Cost      : {cost}")
            print(f"  Nodes Explored  : {self.nodes_explored}")
            print(f"  Search Time     : {elapsed:.6f} seconds")
            print(f"\n  Visited Sequence (exploration order):")
            print(f"    {' → '.join(self.visited_sequence)}")

        print("=" * 60)
        return path, cost

In [44]:
# ============================================================
# PERFORMANCE COMPARISON: IDA* vs A* on both test cases
# ============================================================

def compare_algorithms(graph, source, destination, case_name):
    """Run both IDA* and A* on the same graph and compare results."""
    print("\n" + "#" * 60)
    print(f"  PERFORMANCE COMPARISON — {case_name}")
    print("#" * 60)

    # IDA*
    ida_solver = IDAStarSolver(graph)
    ida_path, ida_cost = ida_solver.print_results(source, destination)

    # A*
    astar_solver = AStarSolver(graph)
    astar_path, astar_cost = astar_solver.print_results(source, destination)

    # Summary table
    print("\n" + "=" * 60)
    print("  COMPARISON SUMMARY")
    print("=" * 60)
    print(f"  {'Metric':<25} {'IDA*':<15} {'A*':<15}")
    print(f"  {'-'*25} {'-'*15} {'-'*15}")

    ida_path_str = ' → '.join(ida_path) if ida_path else "N/A"
    astar_path_str = ' → '.join(astar_path) if astar_path else "N/A"

    print(f"  {'Optimal Cost':<25} {ida_cost:<15} {astar_cost:<15}")
    print(f"  {'Nodes Explored':<25} {ida_solver.nodes_explored:<15} {astar_solver.nodes_explored:<15}")
    print(f"  {'Path':<25} {ida_path_str}")
    print(f"  {'':<25} {astar_path_str}")
    print(f"  {'Memory Usage':<25} {'O(bd)':<15} {'O(b^d)':<15}")
    print("=" * 60)


# Run comparison on Case 1
compare_algorithms(graph1, "MG_Road", "Yelahanka", "Case 1: MG_Road → Yelahanka")

# Run comparison on Case 2
compare_algorithms(graph2, "A", "E", "Case 2: A → E")


############################################################
  PERFORMANCE COMPARISON — Case 1: MG_Road → Yelahanka
############################################################

  IDA* SEARCH — SMART WASTE COLLECTION ROBOT AGENT
  Source      : MG_Road
  Destination : Yelahanka
------------------------------------------------------------

  Optimal Path Found!
  Path            : MG_Road → Electronic_City → Whitefield → Yelahanka
  Total Cost      : 8
  Nodes Explored  : 7
  Search Time     : 0.000037 seconds

  Visited Sequence (exploration order):
    MG_Road → Electronic_City → Whitefield → MG_Road → Electronic_City → Whitefield → Yelahanka

  A* SEARCH — ALTERNATE APPROACH
  Source      : MG_Road
  Destination : Yelahanka
------------------------------------------------------------

  Optimal Path Found!
  Path            : MG_Road → Electronic_City → Whitefield → Yelahanka
  Total Cost      : 8
  Nodes Explored  : 5
  Search Time     : 0.000025 seconds

  Visited Sequence (explor

## 9. Error Handling Demonstration

Demonstrating robustness: invalid nodes, disconnected graphs, negative weights.

In [45]:
# ============================================================
# ERROR HANDLING & EDGE CASES
# ============================================================

print("=" * 60)
print("  ERROR HANDLING DEMONSTRATIONS")
print("=" * 60)

# Test 1: Invalid source node
print("\n--- Test 1: Invalid source node ---")
try:
    solver_err = IDAStarSolver(graph1)
    solver_err.solve("InvalidLocation", "Yelahanka")
except KeyError as e:
    print(f"  Caught error: {e}")

# Test 2: Invalid destination node
print("\n--- Test 2: Invalid destination node ---")
try:
    solver_err = IDAStarSolver(graph1)
    solver_err.solve("MG_Road", "NonExistent")
except KeyError as e:
    print(f"  Caught error: {e}")

# Test 3: Negative edge weight
print("\n--- Test 3: Negative edge weight ---")
try:
    bad_graph = WasteManagementGraph()
    bad_graph.add_edge("X", "Y", -5)
except ValueError as e:
    print(f"  Caught error: {e}")

# Test 4: Disconnected graph (unreachable destination)
print("\n--- Test 4: Disconnected graph ---")
disconnected = WasteManagementGraph()
disconnected.add_edge("P", "Q", 2)
disconnected.add_edge("R", "S", 3)  # separate component
solver_disc = IDAStarSolver(disconnected)
solver_disc.print_results("P", "S")

# Test 5: Battery constraint preventing path
print("\n--- Test 5: Battery constraint ---")
limited = WasteManagementGraph(battery_capacity=5)
limited.add_edge("Start", "Mid", 3)
limited.add_edge("Mid", "End", 4)  # total cost = 7 > capacity 5
solver_lim = IDAStarSolver(limited)
solver_lim.print_results("Start", "End")

print("\nAll error handling tests passed!")

  ERROR HANDLING DEMONSTRATIONS

--- Test 1: Invalid source node ---
  Caught error: "Location 'InvalidLocation' not found in the waste management network. Available locations: ['Electronic_City', 'Hebbal', 'Jayanagar', 'Koramangala', 'MG_Road', 'Whitefield', 'Yelahanka']"

--- Test 2: Invalid destination node ---
  Caught error: "Location 'NonExistent' not found in the waste management network. Available locations: ['Electronic_City', 'Hebbal', 'Jayanagar', 'Koramangala', 'MG_Road', 'Whitefield', 'Yelahanka']"

--- Test 3: Negative edge weight ---
  Caught error: Edge weight must be non-negative, got -5

--- Test 4: Disconnected graph ---

  IDA* SEARCH — SMART WASTE COLLECTION ROBOT AGENT
  Source      : P
  Destination : S
------------------------------------------------------------

  No path exists from 'P' to 'S'.
  Nodes explored : 2

--- Test 5: Battery constraint ---

  IDA* SEARCH — SMART WASTE COLLECTION ROBOT AGENT
  Source      : Start
  Destination : End
-----------------

## 10. Conclusion

### Summary of Results

| Aspect | Details |
|--------|---------|
| **Algorithm** | IDA* (Iterative Deepening A*) — combines A*'s optimality with IDDFS's memory efficiency |
| **Heuristic** | Minimum hop count × Minimum edge cost (admissible and consistent) |
| **Cost Function** | $f(n) = g(n) + h(n)$, where $g(n)$ = path cost, $h(n)$ = heuristic estimate |
| **Optimality** | Guaranteed (admissible heuristic) |
| **Memory** | $O(bd)$ — linear in solution depth |
| **Constraints** | Battery/fuel capacity enforced during search |

### Key Findings
1. IDA* successfully finds optimal routes in both Bengaluru and abstract graph test cases
2. The min-hop × min-edge-cost heuristic provides effective pruning, reducing nodes explored
3. Battery constraints are properly enforced, preventing infeasible routes
4. A* explores fewer nodes but uses exponentially more memory than IDA*
5. Both algorithms find the same optimal path, confirming correctness

### Alternate Approach Verdict
For Bengaluru's waste management network (hundreds to thousands of nodes), **A* is practical** due to manageable graph size. For city-scale routing with millions of intersections, **IDA* is preferred** due to its linear memory footprint.